# HireMind — one focused experiment: does DistilBERT + Knowledge + evidence need more training? (Colab GPU)

**Why this experiment.** DistilBERT + Knowledge v1 + B4 evidence (D0) gave the best single result so far (seed 44: accuracy 60.93, Good Fit recall 66.09) but a very different result on seed 43 (51.55 / 27.34). In almost every DistilBERT run the best epoch was the **last** one (epoch 3), which suggests the model was still learning when training stopped. If so, more epochs should raise the average **and** make seeds agree more.

**What it runs (6 runs, about 15–25 min each on a T4):** the same model, input, labels and settings as the GitHub runs, with **3 epochs vs 5 epochs**, seeds 42, 43, 44. Both arms run on the same GPU, so the comparison is fair (GPU numbers are not mixed with the CPU runs). It also saves class probabilities, so the last cell can show the **ensemble** (average of the 3 seeds) for each arm.

**How to read it.** 5 epochs helps only if the **mean** goes up on accuracy and Good Fit recall and the seeds spread less. One good seed is not enough. Validation only; the test set is never downloaded. Outputs hold scores and row IDs only.

Use a **second Google account** if the first one is busy with the M8 notebook (Colab gives one GPU per account).

In [ ]:
# 1) Runtime check
import torch, platform
print("python", platform.python_version(), "| torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU is very slow for DistilBERT: choose Runtime > Change runtime type > GPU)")

In [ ]:
# 2) Settings
SEEDS = [42, 43, 44]
EPOCH_ARMS = [3, 5]
DRIVE_DIR = "/content/drive/MyDrive/HireMind_colab_epochs"

In [ ]:
# 3) Install the pinned packages (Colab's own CUDA torch is kept)
!pip -q install transformers==4.51.3 pandas==2.2.3 scikit-learn==1.6.1 huggingface-hub==0.30.2 scipy

In [ ]:
# 4) Pinned code: team data loader, team Knowledge v1 code, and the exact experiment scripts used on GitHub
%cd /content
!rm -rf _dl _s2 _m5 _m8 _plan _plan _plan _plan
!git clone -q https://github.com/Hams770/HireMind-DL.git _dl && git -C _dl checkout -q 1f22f34462c400f884bb0a00f061dbea0bd687d5
!git clone -q https://github.com/Hams770/Senior-2.git _s2 && git -C _s2 checkout -q ec28dfc85e618a737cb192cbf959714cfee89aa5
!(test -d _plan || git clone -q --depth 1 https://github.com/HIREMINDPROJECT/HireMind-Plan.git _plan) && cp -r _plan/code/m5-distilbert-plan _m5 && cp _plan/A5_label_overlay_D1/train_labels_v1.csv _m5/phase_c2_m5/e2mods/
!(test -d _plan || git clone -q --depth 1 https://github.com/HIREMINDPROJECT/HireMind-Plan.git _plan) && mkdir -p _m8 && cp -r _plan/code/m8-distilbert/m8 _m8/ && cp _plan/code/m8-seeds/m8/m8_distilbert.py _m8/m8/
!echo "001212500c11a2d471845272718bda0e458968293d551d4979d6331226c178c5  _m5/phase_c2_m5/e2mods/train_labels_v1.csv" | sha256sum -c

In [ ]:
# 5) GPU patch + two small additions: epochs from M5_EPOCHS, and class probabilities in the predictions file
from pathlib import Path
src = Path("/content/_m5/phase_c2_m5/c_stage2_m5_distilbert.py").read_text()
edits = [
    ("EPOCHS, BATCH, LR = 3, 16, 3e-5", 'EPOCHS, BATCH, LR = int(os.environ.get("M5_EPOCHS", "3")), 16, 3e-5'),
    ("Vi, Vm = enc(va, L)\n", "Vi, Vm = enc(va, L)\n" + 'DEV = "cuda" if torch.cuda.is_available() else "cpu"; print("device", DEV, "epochs", EPOCHS, flush=True)\n'
     "model.to(DEV); Xi, Xm, Vi, Vm, ytr = Xi.to(DEV), Xm.to(DEV), Vi.to(DEV), Vm.to(DEV), ytr.to(DEV)\n"),
    ("            preds.append(model(input_ids=Vi[off:off + 32], attention_mask=Vm[off:off + 32]).logits.argmax(-1))\n",
     "            _lg = model(input_ids=Vi[off:off + 32], attention_mask=Vm[off:off + 32]).logits; preds.append(_lg.argmax(-1)); probs.append(_lg.softmax(-1))\n"),
    ("    p = torch.cat(preds).numpy()\n", "    p = torch.cat(preds).cpu().numpy(); P = torch.cat(probs).cpu().numpy()\n"),
    ("best_row = dict(epoch=ep + 1, step=step, metrics=m, cm=cm, p=p, inf_ms=inf_ms)", "best_row = dict(epoch=ep + 1, step=step, metrics=m, cm=cm, p=p, P=P, inf_ms=inf_ms)"),
    ('pd.DataFrame({"row_uid": va.row_uid, "true_id": yva, f"M5_distilbert_{TAG}": best_row["p"]})',
     'pd.DataFrame({"row_uid": va.row_uid, "true_id": yva, f"M5_distilbert_{TAG}": best_row["p"], "p_no": best_row["P"][:, 0], "p_potential": best_row["P"][:, 1], "p_good": best_row["P"][:, 2]})'),
    ("best_state = {k: v.detach().clone() for", "best_state = {k: v.detach().cpu().clone() for"),
]
for a, b in edits:
    assert src.count(a) == 1, a[:60]
    src = src.replace(a, b)
import re
m = re.search(r"\n(\s*)preds = \[\]\n", src); assert m, "preds init not found"
src = src.replace(m.group(0), m.group(0).rstrip("\n") + "; probs = []\n", 1)
Path("/content/_m5/phase_c2_m5/m5_epochs_colab.py").write_text(src)
print("patched copy ready")

In [ ]:
# 6) Run 6 runs (finished ones in Drive are skipped, so you can re-run after a disconnect)
import os, subprocess, sys, glob, shutil
import pandas as pd
from google.colab import drive
drive.mount("/content/drive"); os.makedirs(DRIVE_DIR, exist_ok=True)
for ep in EPOCH_ARMS:
    for seed in SEEDS:
        name = f"kev_D0_ep{ep}_s{seed}"; done = f"{DRIVE_DIR}/{name}/C_stage2_m5_validation.csv"
        if os.path.exists(done): print("skip", name); continue
        OUT = f"/content/out/{name}"; os.makedirs(OUT, exist_ok=True); print("\n=== running", name, flush=True)
        env = dict(os.environ, M5_SEED=str(seed), M5_EPOCHS=str(ep), M5_EVIDENCE="1", M5_E2_DIR="/content/_m5/phase_c2_m5/e2mods",
                   M5_KNOWLEDGE_DIR="/content/_s2/evaluation/knowledge")
        p = subprocess.Popen([sys.executable, "/content/_m5/phase_c2_m5/m5_epochs_colab.py", "/content/_dl", OUT, "384"], env=env,
                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, cwd="/content")
        with open(f"{OUT}/full_log.txt", "w") as log:
            for line in p.stdout:
                log.write(line)
                if any(k in line for k in ("device", "evidence", "epoch", "Error", "Traceback")): print(line, end="")
        if p.wait() != 0: print("FAILED", name); continue
        for f in glob.glob(f"{OUT}/*.pt"): os.remove(f)
        shutil.copytree(OUT, f"{DRIVE_DIR}/{name}", dirs_exist_ok=True)

In [ ]:
# 7) Compare 3 vs 5 epochs: mean ± SD over seeds, best epoch, and the 3-seed ensemble
import numpy as np
def met(y, p):
    cm = np.zeros((3, 3), int)
    for a, b in zip(y, p): cm[a, b] += 1
    tp = np.diag(cm); pr = tp / np.maximum(cm.sum(0), 1); rc = tp / cm.sum(1); f = np.where(pr + rc > 0, 2 * pr * rc / np.maximum(pr + rc, 1e-9), 0)
    return dict(accuracy=100 * tp.sum() / cm.sum(), macro_f1=100 * f.mean(), good_fit_recall=100 * rc[2], good_fit_precision=100 * pr[2], potential_recall=100 * rc[1])
rows, ens = [], []
for ep in EPOCH_ARMS:
    Ps, y = [], None
    for seed in SEEDS:
        d = f"{DRIVE_DIR}/kev_D0_ep{ep}_s{seed}"
        if not os.path.exists(f"{d}/C_stage2_m5_validation.csv"): continue
        r = pd.read_csv(f"{d}/C_stage2_m5_validation.csv").iloc[0]
        rows.append(dict(epochs=ep, seed=seed, best_epoch=int(r.best_epoch), accuracy=r.accuracy_pct, macro_f1=r.macro_f1_pct, good_fit_recall=r.good_fit_recall_pct,
                         good_fit_precision=r.good_fit_precision_pct, potential_recall=r.potential_fit_recall_pct))
        pr = pd.read_csv(f"{d}/C_stage2_M5_validation_predictions.csv").sort_values("row_uid"); y = pr.true_id.to_numpy(); Ps.append(pr[["p_no", "p_potential", "p_good"]].to_numpy())
    if len(Ps) > 1: ens.append(dict(epochs=ep, seeds=len(Ps), **met(y, np.mean(Ps, 0).argmax(1))))
df = pd.DataFrame(rows); df.to_csv(f"{DRIVE_DIR}/epochs_all_runs.csv", index=False)
print("Per run:"); display(df.round(2))
print("Mean ± SD per arm:"); display(df.groupby("epochs")[["accuracy", "macro_f1", "good_fit_recall", "good_fit_precision", "potential_recall"]].agg(["mean", "std"]).round(2))
print("Ensemble of the seeds (average probabilities):"); display(pd.DataFrame(ens).round(2))
pd.DataFrame(ens).to_csv(f"{DRIVE_DIR}/epochs_ensemble.csv", index=False)
print("BERT-Tiny + Knowledge (M4), 8 CPU seeds: accuracy 56.79, macro-F1 48.61, Good Fit recall 36.72, GF precision 43.68, Potential recall 29.68")